# EDA — JobMatch

Exploration du dataset *Job Market Intelligence 2024 - Skills Global Dataset*. L'unité d'analyse est une offre d'emploi (`job_id`). Ce notebook répond aux questions sur les métiers, l'expérience, les plateformes de recrutement, la géographie et les entreprises.

> Exécuter les cellules dans l'ordre. Les graphiques et classements sont calculés à partir des fichiers présents dans `data/archive`.

## 1. Qu'ai-je ?

Nous commençons par charger les offres et décrire leur structure. Les chemins sont résolus depuis la racine du dépôt ou depuis le dossier `notebooks`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

candidats = [Path('data/archive/fact_job_postings.csv'), Path('../data/archive/fact_job_postings.csv')]
chemin_offres = next((p for p in candidats if p.exists()), None)
if chemin_offres is None:
    raise FileNotFoundError('fact_job_postings.csv introuvable. Vérifiez le dossier data/archive.')
offres = pd.read_csv(chemin_offres)
print(f'Fichier : {chemin_offres.resolve()}')
print(f'{offres.shape[0]:,} lignes et {offres.shape[1]} colonnes')
display(offres.head())
display(pd.DataFrame({'type': offres.dtypes.astype(str), 'valeurs distinctes': offres.nunique(dropna=True)}))

### Quelles colonnes sont disponibles ?

Les questions portent ici sur `job_title`, `experience_level`, `platform`, `country`, `region` et `company`. On compte les offres distinctes (`job_id`) pour chaque catégorie afin d'éviter de compter plusieurs fois une même offre.

In [ ]:
colonnes_questions = ['job_id', 'job_title', 'experience_level', 'platform', 'country', 'region', 'company']
manquantes = [c for c in colonnes_questions if c not in offres.columns]
if manquantes:
    raise KeyError(f'Colonnes attendues absentes : {manquantes}')
print('Nombre d’offres (lignes) :', len(offres))
print('Nombre de job_id distincts :', offres['job_id'].nunique())
print('Période brute posting_date :', offres['posting_date'].min(), '→', offres['posting_date'].max() if 'posting_date' in offres else 'colonne absente')

## 2. Puis-je faire confiance aux données ?

Ces contrôles ne corrigent pas la source : ils rendent visibles les limites avant interprétation. Les champs manquants sont examinés, ainsi que les doublons d'identifiant et les catégories vides. Un même titre peut désigner des postes différents; les regroupements par métier reprennent donc le titre tel qu'il est fourni, sans normalisation sémantique.

In [ ]:
qualite = pd.DataFrame({
    'manquants': offres.isna().sum(),
    'manquants_%': (offres.isna().mean() * 100).round(2),
    'valeurs_distinctes': offres.nunique(dropna=True)
}).sort_values('manquants', ascending=False)
display(qualite)
print('Lignes dupliquées entièrement :', offres.duplicated().sum())
print('job_id dupliqués :', offres['job_id'].duplicated().sum())
for col in ['job_title', 'experience_level', 'platform', 'country', 'region', 'company']:
    vides = offres[col].astype('string').str.strip().eq('').sum()
    print(f'{col}: {vides} chaîne(s) vide(s)')

### Vérifications de cohérence

Les dates et salaires sont convertis lorsque ces colonnes existent. Les valeurs non convertibles et les intervalles de salaire inversés sont comptés. La date peut être encodée dans un format non standard : elle est donc traitée avec prudence et n'est pas utilisée pour les classements demandés.

In [ ]:
if 'posting_date' in offres.columns:
    dates = pd.to_datetime(offres['posting_date'], errors='coerce')
    print('Dates non interprétables :', dates.isna().sum(), '/', len(dates))
if {'salary_min_usd', 'salary_max_usd'}.issubset(offres.columns):
    sal_min = pd.to_numeric(offres['salary_min_usd'], errors='coerce')
    sal_max = pd.to_numeric(offres['salary_max_usd'], errors='coerce')
    print('Salaires minimum non numériques :', (sal_min.isna() & offres['salary_min_usd'].notna()).sum())
    print('Salaires maximum non numériques :', (sal_max.isna() & offres['salary_max_usd'].notna()).sum())
    print('Intervalles inversés (min > max) :', (sal_min > sal_max).sum())
print('Attention : la couverture et la représentativité du dataset dépendent de sa source et de sa collecte.')

## 3. Que racontent les variables ?

### Quels sont les métiers les plus représentés ? Combien d'offres par métier ?

Le tableau donne le nombre et la part des offres par intitulé de poste. Le graphique affiche les 15 premiers pour rester lisible; le tableau complet est conservé dans `offres_par_metier`.

In [ ]:
def compter_offres(colonne):
    base = offres.loc[offres[colonne].notna()].copy()
    base[colonne] = base[colonne].astype(str).str.strip()
    base = base[base[colonne].ne('')]
    result = base.groupby(colonne)['job_id'].nunique().sort_values(ascending=False).rename('nombre_offres').to_frame()
    result['part_%'] = (result['nombre_offres'] / offres['job_id'].nunique() * 100).round(2)
    return result

offres_par_metier = compter_offres('job_title')
display(offres_par_metier)
ax = offres_par_metier.head(15).sort_values('nombre_offres').plot.barh(y='nombre_offres', figsize=(10, 6), legend=False, color='#2878B5')
ax.set(title='15 métiers avec le plus d’offres', xlabel='Nombre d’offres distinctes', ylabel='Métier')
plt.tight_layout(); plt.show()

### Quels niveaux d'expérience sont les plus demandés ?

In [ ]:
offres_par_experience = compter_offres('experience_level')
display(offres_par_experience)
ax = offres_par_experience.sort_values('nombre_offres').plot.barh(y='nombre_offres', figsize=(8, 4), legend=False, color='#59A14F')
ax.set(title='Offres par niveau d’expérience', xlabel='Nombre d’offres distinctes', ylabel='Niveau')
plt.tight_layout(); plt.show()

### Quelles plateformes sont le plus utilisées pour trouver les offres ?

La colonne `platform` indique la plateforme de publication/collecte de l'offre. Elle renseigne le canal associé à l'annonce, mais ne permet pas à elle seule de connaître comment un utilisateur individuel a entendu parler de l'offre.

In [ ]:
offres_par_plateforme = compter_offres('platform')
display(offres_par_plateforme)
ax = offres_par_plateforme.sort_values('nombre_offres').plot.barh(y='nombre_offres', figsize=(8, 4), legend=False, color='#F28E2B')
ax.set(title='Offres par plateforme', xlabel='Nombre d’offres distinctes', ylabel='Plateforme')
plt.tight_layout(); plt.show()

### Dans quels pays ou zones trouve-t-on le plus d'offres ?

In [ ]:
offres_par_pays = compter_offres('country')
offres_par_region = compter_offres('region')
display(offres_par_pays)
display(offres_par_region)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
offres_par_pays.head(12).sort_values('nombre_offres').plot.barh(y='nombre_offres', ax=axes[0], legend=False, color='#4E79A7')
axes[0].set(title='12 premiers pays', xlabel='Offres', ylabel='Pays')
offres_par_region.sort_values('nombre_offres').plot.barh(y='nombre_offres', ax=axes[1], legend=False, color='#76B7B2')
axes[1].set(title='Offres par région', xlabel='Offres', ylabel='Région')
plt.tight_layout(); plt.show()

### Quelles entreprises publient le plus d'offres ?

In [ ]:
offres_par_entreprise = compter_offres('company')
display(offres_par_entreprise)
ax = offres_par_entreprise.head(15).sort_values('nombre_offres').plot.barh(y='nombre_offres', figsize=(10, 6), legend=False, color='#B07AA1')
ax.set(title='15 entreprises avec le plus d’offres', xlabel='Nombre d’offres distinctes', ylabel='Entreprise')
plt.tight_layout(); plt.show()

## À retenir

Les tableaux produits donnent les effectifs et les parts calculés sur les offres avec catégorie renseignée. Interpréter les résultats comme une description de ce dataset : ils ne garantissent pas que les mêmes tendances existent sur tout le marché de l'emploi. Les intitulés de postes n'étant pas regroupés par famille professionnelle, des variantes proches peuvent apparaître séparément.